# Thermal SR v5 — distillation + re-parameterisation (Colab, A100)

Trains a better **64/10 model of exactly the shape the app already runs** — same ncnn graph,
same milliseconds per frame — in two stages (plan: `sr_train/PLAN_V5.md`):

1. **Teacher** — a large RRDBNet 64/23 (16.7 M parameters, ~40× the app's model), warm-started
   from RealESRNet_x4plus (Real-ESRGAN's L1 stage, no GAN), trained on v4's data, degradation and
   loss. It never ships; it only has to be good.
2. **Students** — the app's three sizes, **full 64/10, medium 48/10, lite 32/10** (`MODELS` in
   section 5), each warm-started from the model of that size the app ships, trained
   **re-parameterised** (every 3×3 conv as 3×3 + 1×1 + identity) against the GT *and* the teacher's
   output on the same input. Afterwards the branches fold back into a plain SRVGGNetCompact of
   that size. One teacher serves all three.

Then: PSNR table (v4 vs teacher vs v5), the fake-hot-spot check, ncnn export → `thermal_ncnn_v5.zip`,
`…_medium48.zip`, `…_lite32.zip` — the app's **Load SR model (.zip)…** files each under its size.

**Before the long run, do a smoke run:** `SMOKE = True` in section 5 trains 300 iterations per stage
(~15 min) and runs every cell to the end. Then set it back to `False` and *Run all* again.

Runtime → Change runtime type → **A100**. The teacher is ~40× the student; on a T4 it is not worth
it. Time per stage is printed by the trainer (`eta`) after the first few hundred iterations; the
defaults (30k teacher + 60k per student) are the budget of the full recipe; three students
triple the student part — trim `MODELS` to what you need, and check your compute units first.

In [ ]:
!nvidia-smi
import torch
if not torch.cuda.is_available():
    raise SystemExit('No GPU in this runtime. Runtime -> Change runtime type -> T4 GPU, '
                     'then re-run. Training cannot fall back to CPU: Real-ESRGAN calls '
                     '.cuda() directly (USMSharp, the degradation pipeline).')
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name}, {props.total_memory / 1024**3:.1f} GB')
if props.total_memory / 1024**3 < 30:
    print('NOTE: v5 is sized for an A100 (40/80 GB) — the RRDBNet teacher is slow on smaller cards.')
if props.total_memory / 1024**3 < 14:
    print('WARNING: under ~14 GB — drop batch_size to 6 and queue_size to 80 in the '
          'config cell, or stage 2 will run out of memory.')

# ---------------------------------------------------------------------------
USE_DRIVE = True    # recommended for v5: two long stages; False = runtime disk only
# ---------------------------------------------------------------------------

import pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = pathlib.Path('/content/drive/MyDrive/thermal_sr')
else:
    ROOT = pathlib.Path('/content/thermal_sr')
    print('\nNOT using Drive. Everything lives on the runtime disk, which Colab wipes when',
          '\nthe session ends or disconnects — checkpoints included. Run the "save progress"',
          '\ncell now and then to pull the latest checkpoint down to your machine.')

for sub in ('raw', 'val_mag160', 'captures', 'experiments', 'export', 'cache', 'init'):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
print('\nroot:', ROOT)
print('raw sources:', [p.name for p in (ROOT / 'raw').iterdir() if p.is_dir()])

## 2. Install Real-ESRGAN + this project's pipeline

Two things need patching before anything installs, both the same Python 3.13 issue:
`basicsr`'s and Real-ESRGAN's `setup.py` read their version with `exec()` and then
`locals()['__version__']`, which PEP 667 broke — you get `KeyError: '__version__'`.
So we patch basicsr's sdist before installing it, and skip Real-ESRGAN's `setup.py`
entirely by running the trainer from the repo directory instead of installing it.

In [ ]:
%cd /content
!rm -rf Real-ESRGAN MAG160_ThermalCam
!git clone -q https://github.com/xinntao/Real-ESRGAN.git
!git clone -q https://github.com/delphicchen/MAG160_ThermalCam.git
%cd /content/Real-ESRGAN
!cp -r /content/MAG160_ThermalCam/sr_train/options /content/MAG160_ThermalCam/sr_train/thermal_arch /content/MAG160_ThermalCam/sr_train/scripts .

# runtime deps (torch/torchvision/opencv/scipy ship with Colab)
!pip -q install addict yapf lmdb future tqdm onnx onnxsim ncnn pnnx tb-nightly

# --- basicsr ---------------------------------------------------------------
# basicsr's setup.py reads its version with exec() + locals()['__version__'], which
# PEP 667 broke on Python 3.13. pip cannot even *download* it with --no-binary,
# because that runs setup.py egg_info first — so fetch the sdist straight from PyPI,
# patch it, then install from the patched directory.
import json, pathlib, shutil, subprocess, sys, tarfile, urllib.request

work = pathlib.Path('/tmp/bs')
shutil.rmtree(work, ignore_errors=True)
work.mkdir(parents=True)

meta = json.load(urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json'))
url = next(u['url'] for u in meta['urls'] if u['packagetype'] == 'sdist')
tgz = work / 'basicsr.tar.gz'
urllib.request.urlretrieve(url, tgz)
with tarfile.open(tgz) as t:
    t.extractall(work, filter='data')
pkg = next(p for p in work.iterdir() if p.is_dir())

sp = pkg / 'setup.py'
src = sp.read_text()
before = src
src = src.replace("exec(compile(f.read(), version_file, 'exec'))",
                  "_ns = {}\n        exec(compile(f.read(), version_file, 'exec'), _ns)")
src = src.replace("return locals()['__version__']", "return _ns['__version__']")
assert src != before, 'setup.py did not match the expected pattern — inspect it manually'
sp.write_text(src)
print('patched', sp)

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(pkg), '--no-deps'],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
r.check_returncode()

# --- torchvision >= 0.17 moved functional_tensor; basicsr still imports the old path ---
# Find the package WITHOUT importing it: importing basicsr executes the very line that
# is broken, so `import basicsr` cannot come before the patch.
import importlib.util
spec = importlib.util.find_spec('basicsr')
assert spec and spec.origin, 'basicsr is not importable — the install above failed'
pkg_dir = pathlib.Path(spec.origin).parent
hit = []
for f in pkg_dir.rglob('*.py'):          # degradations.py is the known one; catch any other
    t = f.read_text()
    if 'functional_tensor' in t:
        f.write_text(t.replace('torchvision.transforms.functional_tensor',
                               'torchvision.transforms.functional'))
        hit.append(f.name)
print('patched:', hit)

# --- realesrgan/version.py is generated by setup.py, which we deliberately skip ---
# realesrgan/__init__.py does `from .version import *`, so the package needs one.
ver = pathlib.Path('realesrgan/version.py')
if not ver.exists():
    v = pathlib.Path('VERSION').read_text().strip() if pathlib.Path('VERSION').exists() else '0.3.0'
    ver.write_text(f"__version__ = '{v}'\n__gitsha__ = 'unknown'\n"
                   f"version_info = tuple(int(x) for x in '{v}'.split('.')[:3])\n")
    print('wrote', ver, v)

# --- register our degradation models with BasicSR's registry ---
# `python realesrgan/train.py` puts realesrgan/ first on sys.path, and that directory
# has its own `archs` package — a top-level `archs` of ours would be shadowed by it
# (ModuleNotFoundError: No module named 'archs.thermal_degradation'). Hence thermal_arch,
# plus an explicit repo-root insert so the import does not depend on PYTHONPATH.
p = pathlib.Path('realesrgan/train.py')
s = p.read_text()
if 'thermal_v5' not in s:
    s = s.replace("import os.path as osp",
                  "import os.path as osp\n"
                  "import sys as _sys\n"
                  "_sys.path.insert(0, osp.dirname(osp.dirname(osp.abspath(__file__))))\n"
                  "import thermal_arch.thermal_degradation  # noqa: F401\n"
                  "import thermal_arch.thermal_v5  # noqa: F401", 1)
    p.write_text(s)
print(pathlib.Path('realesrgan/train.py').read_text().splitlines()[:8])

# --- verify before moving on ----------------------------------------------
import basicsr
from basicsr.data.degradations import circular_lowpass_kernel  # noqa: F401
from basicsr.utils.registry import MODEL_REGISTRY
import thermal_arch.thermal_degradation  # noqa: F401
import thermal_arch.thermal_v5  # noqa: F401
from basicsr.utils.registry import ARCH_REGISTRY
import torch
print('basicsr', basicsr.__version__, '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())
print('registered:', [k for k in MODEL_REGISTRY._obj_map if 'Thermal' in k],
      [k for k in ARCH_REGISTRY._obj_map if k == 'RepSRVGG'])
assert 'ThermalDistillModel' in MODEL_REGISTRY._obj_map and 'RepSRVGG' in ARCH_REGISTRY._obj_map

## 3. Get thermal imagery — no account needed

FLIR's own ADAS download and the PBVS challenge data both need registration. These
Hugging Face mirrors do not — each was checked to be public and ungated:

| source | images | size | note |
|---|---|---|---|
| `jsonhash/FLIR_aligned` | ~5.1k thermal @ **640×512** | 1.4 GB zip | FLIR ADAS aligned pairs; we take only `*_PreviewData.jpeg` |
| `vision-cidis/CIDIS-dataset` (GitHub) | 700 train + 200 val @ **640×448** | ~0.8 GB | PBVS TISR 2024/25 benchmark; grey, milder AGC than FLIR's previews. No licence file — its README asks for a citation (Rivadeneira et al., 2024) |
| `LibreYOLO/flir-camera-objects` | 13.6k @ 640×640 | ~1 GB | same FLIR ADAS via Roboflow, **stretched** to square — off by default |
| `Kiuyha/hit-uav-thermal-human-detection` | 4.9k @ 640×640 | ~300 MB | drone thermal, also a Roboflow export (`.rf.`), stretched — off by default |

FLIR aligned + CIDIS train is the default. CIDIS val stays out of training and becomes the
section 8 check set when you have no real captures. Raw images land in `/content/raw` (local disk — Drive
is far too slow for thousands of small files) and the zip is cached on Drive so a
reconnect does not re-download it. Anything you put in `Drive/thermal_sr/raw/` is picked
up as an extra source.


In [ ]:
import pathlib, shutil, zipfile
from huggingface_hub import hf_hub_download, snapshot_download

EXTRA_SOURCES = False   # True also pulls the two Roboflow sets (~1.3 GB, stretched 640x640)

raw = pathlib.Path('/content/raw'); raw.mkdir(parents=True, exist_ok=True)
cache = ROOT / 'cache'; cache.mkdir(exist_ok=True)

# --- FLIR aligned: native 640x512 thermal, the best of the three ---
zp = cache / 'flir_aligned.zip'
if not zp.exists():
    got = hf_hub_download('jsonhash/FLIR_aligned', 'aligned.zip',
                          repo_type='dataset', local_dir='/content/hf')
    shutil.copy(got, zp)          # keep on Drive: survives a disconnect
    print('cached', zp)

dst = raw / 'flir_aligned'; dst.mkdir(exist_ok=True)
if not any(dst.iterdir()):
    with zipfile.ZipFile(zp) as z:
        members = [n for n in z.namelist() if n.endswith('_PreviewData.jpeg')]
        print(f'extracting {len(members)} thermal frames…')
        for n in members:
            with z.open(n) as src, open(dst / pathlib.Path(n).name, 'wb') as out:
                shutil.copyfileobj(src, out)
print('flir_aligned:', len(list(dst.glob('*.jpeg'))), 'images')

if EXTRA_SOURCES:
    for repo, sub in (('LibreYOLO/flir-camera-objects', 'flir_roboflow'),
                      ('Kiuyha/hit-uav-thermal-human-detection', 'hit_uav')):
        d = raw / sub
        if not d.exists():
            snapshot_download(repo, repo_type='dataset', local_dir=str(d),
                              allow_patterns=['*/images/*.jpg'])
        print(sub, len(list(d.rglob('*.jpg'))), 'images')

# --- CIDIS: PBVS TISR 2024/25 benchmark, public on GitHub, no account -------------------
# Only the thermal train/val folders are checked out (sparse, blobless). train joins the
# crops; val is kept apart — section 8 uses it as the check set if you have no captures.
import subprocess
cidis = pathlib.Path('/content/cidis')
if not (cidis / 'dataset/thermal/train').exists():
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--filter=blob:none', '--sparse',
                    'https://github.com/vision-cidis/CIDIS-dataset.git', str(cidis)], check=True)
    subprocess.run(['git', '-C', str(cidis), 'sparse-checkout', 'set',
                    'dataset/thermal/train', 'dataset/thermal/val'], check=True)
d = raw / 'cidis'
if not d.exists():
    shutil.copytree(cidis / 'dataset/thermal/train', d)
print('cidis:', len(list(d.glob('*.bmp'))), 'train images;',
      len(list((cidis / 'dataset/thermal/val').glob('*.bmp'))), 'val images kept apart')

# --- anything you dropped on Drive yourself ---
if any((ROOT / 'raw').iterdir()):
    shutil.copytree(ROOT / 'raw', raw / 'user', dirs_exist_ok=True)
    print('user:', len(list((raw / 'user').rglob('*'))), 'entries')

!du -sh /content/raw/*


### Cut them into HR crops

640×512 and 640×640 sources give a single 480px tile at stride 360, so the stride is
shortened to overlap them into several — that is what gets the crop count up.


In [ ]:
!PYTHONPATH=/content/Real-ESRGAN python scripts/prepare_thermal_dataset.py \
    --raw /content/raw \
    --out datasets/thermal_hr \
    --meta datasets/meta_info/thermal_hr.txt \
    --crop 480 --stride 160


In [ ]:
# A run with a handful of images memorises them and teaches you nothing — check before
# spending hours on it.
import pathlib
n = len(pathlib.Path('datasets/meta_info/thermal_hr.txt').read_text().split())
print(f'{n} HR crops')
if n < 200:
    raise SystemExit('Too few crops to train on. Is Drive/thermal_sr/raw/ actually '
                     'populated with HR thermal images of at least 480x480? Sub-folders '
                     'or loose files both work.')
if n < 5000:
    print('WARNING: under ~5k crops the GAN stage will overfit — more sources would help.')


## 5. v5 configs, warm starts

`SMOKE = True` runs 300 iterations per stage — use it first. The sensor ranges are v4's.
`KD_WEIGHT` is the teacher's share of the student loss (0.5 = half the GT's weight).

Warm starts made here:
- teacher ← `RealESRNet_x4plus.pth`, RGB folded into one channel (`scripts/grey_teacher_init.py`);
- student ← v4 rebuilt from the app's bundled ncnn model (`scripts/ncnn_to_pth.py`), unfolded into
  the re-parameterised layout so it starts as v4 exactly (`scripts/rep_fold.py --init`, self-checking).

In [ ]:
import os, pathlib, shutil, time, urllib.request, yaml, torch
os.chdir('/content/Real-ESRGAN')

# ---------------------------------------------------------------------------
SMOKE = False                          # True: 300 iterations per stage, end-to-end check
MODELS = ['full', 'medium', 'lite']    # students to train, in this order; one teacher for all
TEACHER_ITER, STUDENT_ITER = 30000, 60000
KD_WEIGHT = 0.5                        # teacher term vs the GT term (1.0) in the student loss
SENSOR = dict(
    sensor_span_c=[0.5, 40.0], sensor_noise_c=[0.02, 0.20], sensor_noise_log=True,
    sensor_stripe_c=[0.0, 0.10], sensor_map_c=[0.0, 0.05],
)
# ---------------------------------------------------------------------------
if SMOKE:
    TEACHER_ITER, STUDENT_ITER = 300, 300
RECIPE = f'v5-rrdb23-rep-kd{KD_WEIGHT}' + ('-smoke' if SMOKE else '')
WIDTH = {'full': 64, 'medium': 48, 'lite': 32}
ASSET = {'full': 'model', 'medium': 'model-medium', 'lite': 'model-lite'}   # the app's bundled models
def stage_of(m):
    return 'thermal_v5_student' + {'full': '', 'medium': '_medium48', 'lite': '_lite32'}[m]

GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
if GPU_GB >= 30:
    BATCH, GT, QUEUE, WORKERS = 12, 256, 180, min(8, os.cpu_count() or 2)
else:
    BATCH, GT, QUEUE, WORKERS = 6, 192, 96, 2
print(f'{torch.cuda.get_device_name(0)} ({GPU_GB:.0f} GB): batch {BATCH}, gt {GT}, queue {QUEUE}, '
      f'workers {WORKERS} | teacher {TEACHER_ITER}, student {STUDENT_ITER}, KD {KD_WEIGHT}')

def tune(src, dst, total_iter, model=None):
    o = yaml.safe_load(open(src))
    if model:
        o['name'] = stage_of(model)
        o['network_g']['num_feat'] = WIDTH[model]
        o['path']['pretrain_network_g'] = f'experiments/pretrained_models/v5_rep_init_{model}.pth'
    o.update(SENSOR)
    o['gt_size'] = GT
    o['queue_size'] = QUEUE
    t = o['datasets']['train']
    t['gt_size'], t['batch_size_per_gpu'], t['num_worker_per_gpu'] = GT, BATCH, WORKERS
    o['train']['total_iter'] = total_iter
    o['train']['scheduler']['milestones'] = [int(total_iter * 0.6)]
    if 'distill_opt' in o['train']:
        o['train']['distill_opt']['loss_weight'] = KD_WEIGHT
    o['logger']['save_checkpoint_freq'] = min(2000, total_iter)
    yaml.safe_dump(o, open(dst, 'w'), sort_keys=False)

tune('options/03_teacher_rrdb_x4.yml', 'options/colab_v5_teacher.yml', TEACHER_ITER)
for m in MODELS:
    tune('options/04_distill_rep_x4.yml', f'options/colab_v5_student_{m}.yml', STUDENT_ITER, m)

!rm -rf experiments && ln -sfn {ROOT}/experiments experiments
pm = pathlib.Path('experiments/pretrained_models'); pm.mkdir(parents=True, exist_ok=True)
f = pm / 'RealESRNet_x4plus.pth'
if not f.exists() or f.stat().st_size < 10_000_000:
    print('downloading RealESRNet_x4plus.pth')
    urllib.request.urlretrieve(
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.1/RealESRNet_x4plus.pth', f)
assert f.stat().st_size > 10_000_000, 'RealESRNet_x4plus.pth is missing or truncated'

!PYTHONPATH=/content/Real-ESRGAN python scripts/grey_teacher_init.py \
    --src experiments/pretrained_models/RealESRNet_x4plus.pth \
    --out experiments/pretrained_models/RealESRNet_x4plus-grey.pth

# each student starts from the model of its size the app ships (rebuilt from ncnn, fp16-rounded);
# a .pth in <root>/init/<full|medium|lite>/ wins when present (e.g. v4's own checkpoint)
for m in MODELS:
    own = sorted((ROOT / 'init' / m).glob('*.pth')) if (ROOT / 'init' / m).is_dir() else []
    v4 = own[-1] if own else pm / f'thermal_v4_{m}_from_ncnn.pth'
    if not own:
        bundled = f'/content/MAG160_ThermalCam/android2/app/src/main/assets/{ASSET[m]}/thermal_160x120_fp16'
        get_ipython().system(f'python scripts/ncnn_to_pth.py --param {bundled}.param --bin {bundled}.bin --out {v4}')
    print(f'{m} ({WIDTH[m]}/10) warm start:', v4)
    get_ipython().system(f'PYTHONPATH=/content/Real-ESRGAN python scripts/rep_fold.py --init '
                         f'--src "{v4}" --out {pm}/v5_rep_init_{m}.pth')
    assert (pm / f'v5_rep_init_{m}.pth').exists(), f'rep_fold.py --init failed for {m} — see above'

for stage in ['thermal_v5_teacher'] + [stage_of(m) for m in MODELS]:
    d = pathlib.Path('experiments') / stage
    mark = pathlib.Path('experiments') / f'{stage}.recipe'
    if d.exists() and (not mark.exists() or mark.read_text().strip() != RECIPE):
        dst = pathlib.Path('experiments/_archive') / f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}'
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(d), str(dst))
        print(f'archived a run from another recipe: {d} -> {dst}')
    mark.write_text(RECIPE + '\n')
print('recipe:', RECIPE, '| students:', MODELS)
!ls -la experiments/pretrained_models

## 6. Teacher — RRDBNet 64/23

Same data, degradation and loss as v4's stage 1. Re-run after a disconnect: `--auto_resume`.

In [ ]:
!PYTHONPATH=/content/Real-ESRGAN python realesrgan/train.py -opt options/colab_v5_teacher.yml --auto_resume
!ls -la experiments/thermal_v5_teacher/models/ || echo 'the teacher produced no checkpoint'

## 7. Students — re-parameterised, distilled from the teacher

One after another, in `MODELS` order. Each starts from the app's model of its size exactly; the loss
is L1+spectrum+gradient against the GT plus the same against the teacher's output (`l_pix` and
`l_kd` in the log). Re-run after a disconnect: finished students are skipped by `--auto_resume`.

In [ ]:
import pathlib, re, yaml

def iter_of(q):
    m = re.findall(r'\d+', q.stem)
    return int(m[-1]) if m else -1

def newest(d):
    cks = sorted(pathlib.Path(d).glob('net_g_*.pth'), key=iter_of)
    numbered = [c for c in cks if iter_of(c) >= 0]
    return (numbered or cks or [None])[-1]

teacher = newest('experiments/thermal_v5_teacher/models')
if teacher is None:
    raise SystemExit('No teacher checkpoint yet — run section 6 to completion first.')
print('teacher:', teacher)
for m in MODELS:
    opt = f'options/colab_v5_student_{m}.yml'
    o = yaml.safe_load(open(opt))
    o['path']['pretrain_network_t'] = str(teacher)
    yaml.safe_dump(o, open(opt, 'w'), sort_keys=False)
    print(f'\n===== student {m} ({WIDTH[m]}/10) -> experiments/{stage_of(m)}')
    get_ipython().system(f'PYTHONPATH=/content/Real-ESRGAN python realesrgan/train.py -opt {opt} --auto_resume')
    get_ipython().system(f'ls experiments/{stage_of(m)}/models/ | tail -3')

### Save progress (run whenever, especially without Drive)

Zips the newest teacher and student checkpoints and downloads them.

In [ ]:
import pathlib, re, shutil
def iter_of(q):
    m = re.findall(r'\d+', q.stem)
    return int(m[-1]) if m else -1
out = pathlib.Path('/content/progress_v5'); out.mkdir(exist_ok=True)
for stage in ('thermal_v5_teacher', 'thermal_v5_student', 'thermal_v5_student_medium48',
              'thermal_v5_student_lite32'):
    cks = sorted(pathlib.Path(f'experiments/{stage}/models').glob('net_g_*.pth'), key=iter_of)
    if cks:
        shutil.copy(cks[-1], out / f'{stage}_{cks[-1].name}')
        print('saved', stage, cks[-1].name)
shutil.make_archive('/content/thermal_sr_v5_progress', 'zip', out)
from google.colab import files
files.download('/content/thermal_sr_v5_progress.zip')

## 8. Fold, then judge: PSNR and fake hot spots

Each student is folded into a plain SRVGGNetCompact (`scripts/rep_fold.py`, self-checking), then
compared with the app's model of the same size and the teacher on the 200 CIDIS val frames (never
trained on), 4× down and back up. **Ship a v5 size only if** it beats the app's model of that size
on PSNR *and* invents no more peaks than it in the
hallucination check (real 160×120 frames in `<root>/val_mag160/` if you have them, else 20
CIDIS val frames).

In [ ]:
import os, sys, pathlib, re, subprocess
os.chdir('/content/Real-ESRGAN')
ROOT = pathlib.Path(os.readlink('experiments')).parent      # restart-safe
env = dict(os.environ, PYTHONPATH='/content/Real-ESRGAN')

def iter_of(q):
    m = re.findall(r'\d+', q.stem)
    return int(m[-1]) if m else -1
def newest(d):
    cks = sorted(pathlib.Path(d).glob('net_g_*.pth'), key=iter_of)
    numbered = [c for c in cks if iter_of(c) >= 0]
    return (numbered or cks or [None])[-1]

SIZES = {'full': ('', 64), 'medium': ('_medium48', 48), 'lite': ('_lite32', 32)}
teacher = newest('experiments/thermal_v5_teacher/models')
PAIRS = {}      # size -> (the app's current model, folded v5)
for m, (suf, w) in SIZES.items():
    student = newest(f'experiments/thermal_v5_student{suf}/models')
    if student is None:
        continue
    folded = pathlib.Path(f'export/v5{suf}_student_folded.pth')
    subprocess.run([sys.executable, 'scripts/rep_fold.py', '--src', str(student), '--out', str(folded)],
                   env=env, check=True)
    own = sorted((ROOT / 'init' / m).glob('*.pth')) if (ROOT / 'init' / m).is_dir() else []
    PAIRS[m] = (own[-1] if own else pathlib.Path(f'experiments/pretrained_models/thermal_v4_{m}_from_ncnn.pth'), folded)
if not PAIRS:
    raise SystemExit('No student checkpoint — run section 7 first.')
print({m: str(f) for m, (_, f) in PAIRS.items()})

models = [f'{tag}_{m}=srvgg:{ck}' for m, (v4, v5) in PAIRS.items() for tag, ck in (('v4', v4), ('v5', v5))]
models += [f'teacher=rrdb:{teacher}'] if teacher else []
cmd = [sys.executable, 'scripts/eval_psnr.py', '--hr', '/content/cidis/dataset/thermal/val',
       '--csv', 'results/v5_psnr.csv']
for m in models:
    cmd += ['--model', m]
pathlib.Path('results').mkdir(exist_ok=True)
subprocess.run(cmd, env=env, check=True)

# fake hot spots: real captures first, else 20 CIDIS val frames 4x down
val = ROOT / 'val_mag160'; val.mkdir(parents=True, exist_ok=True)
if not any(val.iterdir()):
    import cv2
    val = ROOT / 'val_cidis'; val.mkdir(exist_ok=True)
    if not any(val.iterdir()):
        for s in sorted(pathlib.Path('/content/cidis/dataset/thermal/val').glob('*.bmp'))[:20]:
            g = cv2.imread(str(s), cv2.IMREAD_GRAYSCALE)
            cv2.imwrite(str(val / f'{s.stem}.png'),
                        cv2.resize(g, (g.shape[1] // 4, g.shape[0] // 4), interpolation=cv2.INTER_AREA))
from IPython.display import Image, display
for tag, ck in [(f'{t}_{m}', c) for m, (v4, v5) in PAIRS.items() for t, c in (('v4', v4), ('v5', v5))]:
    o = f'results/val_{tag}'
    print(f'\n===== {tag}: {ck}')
    subprocess.run([sys.executable, 'scripts/hallucination_check.py', '--ckpt', str(ck), '--arch', 'srvgg',
                    '--lr', str(val), '--out', o], env=env)
    for p in sorted(pathlib.Path(o).glob('*_check.png'))[:2]:
        display(Image(str(p)))

## 9. Export → ncnn fp16, verify, download

Each folded size, verified against PyTorch: full at 160×120 / 120×160 as `thermal_ncnn_v5.zip`,
medium and lite at 256×192 / 192×256 (v4's convention; the graphs take any size) as
`thermal_ncnn_v5_medium48.zip` / `thermal_ncnn_v5_lite32.zip`.

In [ ]:
import os, sys, pathlib, shutil, subprocess, zipfile
os.chdir('/content/Real-ESRGAN')
ROOT = pathlib.Path(os.readlink('experiments')).parent
env = dict(os.environ, PYTHONPATH='/content/Real-ESRGAN')
from google.colab import files
done = []
for suf in ('', '_medium48', '_lite32'):
  FOLDED = pathlib.Path(f'export/v5{suf}_student_folded.pth')
  if not FOLDED.exists():
    continue
  sizes = ((160, 120), (120, 160)) if not suf else ((256, 192), (192, 256))
  out = pathlib.Path(f'export/v5{suf}'); out.mkdir(parents=True, exist_ok=True)
  for w, h in sizes:
    base = f'{out}/thermal_x4_{w}x{h}'
    subprocess.run([sys.executable, 'scripts/export_onnx.py', '--ckpt', str(FOLDED), '--arch', 'srvgg',
                    '--size', f'{w}x{h}', '--out', base], env=env, check=True)
    subprocess.run(['bash', 'scripts/convert_ncnn.sh', base, str(w), str(h), f'thermal_{w}x{h}'],
                   env=env, check=True)
    subprocess.run([sys.executable, 'scripts/verify_ncnn.py', '--ref', f'{base}_ref.pt',
                    '--param', f'{out}/thermal_{w}x{h}_fp16.param',
                    '--bin', f'{out}/thermal_{w}x{h}_fp16.bin'], env=env)
  z = pathlib.Path(f'/content/thermal_ncnn_v5{suf}.zip')
  with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(out.glob('thermal_*_fp16.*')):
      zf.write(f, f.name)
  (ROOT / 'export').mkdir(parents=True, exist_ok=True)
  shutil.copy(z, ROOT / 'export' / z.name)
  shutil.copy(FOLDED, ROOT / 'export' / FOLDED.name)
  done.append(z)
  print('packaged', z, f'{z.stat().st_size / 1e6:.2f} MB')
if pathlib.Path('results/v5_psnr.csv').exists():
  shutil.copy('results/v5_psnr.csv', ROOT / 'export' / 'v5_psnr.csv')
if not done:
  raise SystemExit('Run section 8 first — it folds the students.')
for z in done:
  files.download(str(z))

---
### What to send back

The `thermal_ncnn_v5*.zip` packages (load them in the app's drawer to compare with the built-in
models of the same size), the PSNR table printed in section 8 (`v5_psnr.csv` on Drive), and the
`results/val_*/summary.csv` fake-hot-spot numbers. Keep the folded `.pth` files and the teacher
checkpoint on Drive — the teacher can distil more students later without being trained again.